# 01. Underwater Turbidity Degradation Demo

Welcome to the interactive demonstration notebook for the **Turbid Water Degradation Generator**!

This notebook demonstrates how clean underwater imagery undergoes realistic physical degradation using our Python optical engine (`generator/degradation.py`).

---
## 1. The Physics Model: Jaffe-McGlamery & Beer-Lambert Law

Underwater image degradation is governed by light absorption, scattering, and ambient light veiling. Our simulator implements the classic **Jaffe-McGlamery Underwater Image Formation Model**:

AAI(x) = J(x) \cdot t(x) + A \cdot (1 - t(x))AA

### Key Physics Variables:
1. **AJ(x)A**: The clean input image radiance at scene point AxA.
2. **At(x)A**: The 3-channel wavelength-dependent transmission map (At_c \in [0, 1]A), calculated via **Beer-Lambert's Law**:
   AAt_c(x) = \exp\left(-\beta_c \cdot \text{turbidity} \cdot d(x)\right), \quad c \in \{R, G, B\}AA
   - **Red Attenuation (A\beta_r = 3.0A)**: Red light absorbs rapidly in water.
   - **Green Attenuation (A\beta_g = 1.8A)**: Green light attenuates moderately.
   - **Blue Attenuation (A\beta_b = 1.0A)**: Blue light penetrates deepest.
3. **Ad(x)A**: The normalized scene depth map (A0.0 = \text{near}, 1.0 = \text{far}A).
4. **AAA**: The 3-channel ambient backscatter light vector (AAA = [0.10, 0.45, 0.40] × τ + [0.85, 0.90, 0.95] × (1 - τ)).

### Four Optical Effects Applied:
* **Color Attenuation**: Selective decay of longer wavelengths (reds fade first).
* **Ambient Backscatter Haze**: Additive blue-green veil that builds with distance and turbidity.
* **Forward Scattering Blur**: Dynamic Gaussian spatial blur reducing local high-frequency contrast.
* **Marine Snow Particulates**: Suspended sediment particles reflecting ambient light.

---
## 2. Setup Environment & Import Generator Engine

In [ ]:
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to sys.path so generator module can be imported
sys.path.append(os.path.abspath('..'))

from generator.utils import load_image
from generator.degradation import degrade_image

print("Imports and environment setup successful!")

---
## 3. Load a Clean Sample Image

We load a clean underwater fauna image from `data/raw/` to serve as our base scene radiance AJ(x)A.

In [ ]:
# Locate sample images in data/raw/
sample_dir = os.path.join('..', 'data', 'raw')
image_files = sorted(glob.glob(os.path.join(sample_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(sample_dir, '*.png')))

if not image_files:
    raise FileNotFoundError(f"No sample images found in {sample_dir}")

sample_path = image_files[0]
print(f"Loading clean sample scene: {sample_path}")

# Load clean image (RGB format, uint8)
clean_img = load_image(sample_path)
print(f"Image Shape: {clean_img.shape}, Data Type: {clean_img.dtype}")

# Display clean input image
plt.figure(figsize=(6, 5))
plt.imshow(clean_img)
plt.title("Original Clean Scene Radiance J(x)", fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()

---
## 4. Apply Turbidity Progression & Display 5-Image Side-by-Side Plot

We now pass our clean image AJ(x)A through `degrade_image()` at four controlled turbidity levels:
* **A\tau = 0.2A** (Low Turbidity / Mild Attenuation)
* **A\tau = 0.4A** (Moderate Turbidity / Noticeable Haze)
* **A\tau = 0.6A** (High Turbidity / Heavy Backscatter)
* **A\tau = 0.8A** (Severe Turbidity / High Scattering & Noise)

In [ ]:
# Target turbidity levels
turbidity_levels = [0.2, 0.4, 0.6, 0.8]

# List storing (title, image) pairs starting with clean image
results = [("Clear (t=0.0)", clean_img)]

print("Generating degraded synthetic variants...")
for turb in turbidity_levels:
    degraded = degrade_image(clean_img, turbidity_level=turb, depth_mode="gradient")
    results.append((f"Turbid t={turb:.1f}", degraded))
    print(f"  [+] Processed turbidity level t={turb:.1f}")

# Display all 5 images side-by-side using matplotlib
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle("Underwater Optical Turbidity Progression (Jaffe-McGlamery Model)", fontsize=15, fontweight='bold', y=1.03)

for idx, (title, img) in enumerate(results):
    axes[idx].imshow(img)
    axes[idx].set_title(title, fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

---
## 5. Summary & Next Steps

* **Physical Accuracy**: As turbidity A\tauA increases, red light attenuates rapidly while blue-green ambient veil dominates.
* **Dataset Integration**: All generated synthetic variants preserve exact 1-to-1 spatial alignment with ground-truth segmentation masks.
* **CLI & Web App**: Run `python generator/generate.py --all-levels` for bulk dataset generation or `streamlit run generator/app.py` for real-time Web UI exploration.